# 06 — Simulator development sandbox

Iterate on the **physics-based machine simulator** here, in isolation from
Eventstream / Fabric / GPU training.

Workflow:

1. Define the machine's **operating-state FSM** (`OFF`, `STARTUP`, `IDLE`, `PRODUCTION_LIGHT/HEAVY`, `RAMP_UP/DOWN`, `SHUTDOWN`).
2. Define a **physical state** (load, motor temp, bearing temp) with first-order dynamics.
3. **Derive** the 8 sensor channels from physical state (with coupling).
4. Run a long simulation, collect into a DataFrame.
5. Plot: state timeline, sensor traces, sensor correlations.
6. Tune coefficients until the traces "look real", then port back to `simulator-local/simulate_machines.py`.

> Reference plan: `.copilot/PLAN.md` Phase 1.

## 1. Setup

In [ ]:
from __future__ import annotations

import math
import random
from dataclasses import dataclass, field
from enum import Enum

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility while we tune
RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 3)
plt.rcParams["axes.grid"] = True

## 2. Operating-state FSM

States and how they transition. Each state has:
- a **target load** (0..1) the machine asks the actuator to reach,
- a **dwell distribution** (how long it stays before considering a transition),
- a **next-state distribution** (what comes next, with probabilities).

`RAMP_*` are short transitional states between two regimes — the physical load smoothly walks toward the new setpoint.

In [ ]:
class State(str, Enum):
    OFF              = "OFF"
    STARTUP          = "STARTUP"
    IDLE             = "IDLE"
    PRODUCTION_LIGHT = "PRODUCTION_LIGHT"
    PRODUCTION_HEAVY = "PRODUCTION_HEAVY"
    RAMP_UP          = "RAMP_UP"
    RAMP_DOWN        = "RAMP_DOWN"
    SHUTDOWN         = "SHUTDOWN"


@dataclass
class StateSpec:
    target_load: float                       # 0..1
    dwell_s: tuple[float, float]             # (min, max) seconds in this state
    transitions: list[tuple[State, float]]   # (next_state, weight)


# Tunable transition table. Weights are renormalized on use.
STATE_SPECS: dict[State, StateSpec] = {
    State.OFF: StateSpec(
        target_load=0.0,
        dwell_s=(120, 600),                  # 2..10 min off (short for demo)
        transitions=[(State.STARTUP, 1.0)],
    ),
    State.STARTUP: StateSpec(
        target_load=0.10,
        dwell_s=(15, 30),
        transitions=[(State.IDLE, 1.0)],
    ),
    State.IDLE: StateSpec(
        target_load=0.10,
        dwell_s=(60, 300),
        transitions=[
            (State.RAMP_UP,  0.7),
            (State.SHUTDOWN, 0.1),
            (State.IDLE,     0.2),
        ],
    ),
    State.RAMP_UP: StateSpec(
        target_load=0.5,
        dwell_s=(10, 30),
        transitions=[
            (State.PRODUCTION_LIGHT, 0.5),
            (State.PRODUCTION_HEAVY, 0.5),
        ],
    ),
    State.PRODUCTION_LIGHT: StateSpec(
        target_load=0.40,
        dwell_s=(180, 900),
        transitions=[
            (State.RAMP_UP,          0.3),   # to HEAVY
            (State.RAMP_DOWN,        0.4),
            (State.PRODUCTION_LIGHT, 0.3),
        ],
    ),
    State.PRODUCTION_HEAVY: StateSpec(
        target_load=0.85,
        dwell_s=(120, 600),
        transitions=[
            (State.RAMP_DOWN,        0.6),
            (State.PRODUCTION_HEAVY, 0.4),
        ],
    ),
    State.RAMP_DOWN: StateSpec(
        target_load=0.20,
        dwell_s=(10, 30),
        transitions=[
            (State.IDLE,             0.6),
            (State.PRODUCTION_LIGHT, 0.4),
        ],
    ),
    State.SHUTDOWN: StateSpec(
        target_load=0.0,
        dwell_s=(15, 40),
        transitions=[(State.OFF, 1.0)],
    ),
}


def pick_next_state(current: State) -> State:
    spec = STATE_SPECS[current]
    states, weights = zip(*spec.transitions)
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()
    return State(np.random.choice(states, p=weights))


def pick_dwell(state: State) -> float:
    lo, hi = STATE_SPECS[state].dwell_s
    return random.uniform(lo, hi)

## 3. Physical state + sensor model

The physical state drives the sensors:

- `load_actual` — first-order lag toward `target_load` of the current state. Time constant differs between ramps (fast) and steady states (slower).
- `T_motor` — first-order with heating ∝ `load²` and cooling toward ambient. Slow time constant (minutes).
- `T_bearing` — slaved to `T_motor` with extra lag.

Sensors are derived from `(load_actual, T_motor, T_bearing)` plus noise.

| Sensor | Driver |
|---|---|
| `spindle_rpm` | nominal RPM, droops slightly under heavy load |
| `current` | a + b·load |
| `power` | current × constant (≈ V·I) |
| `pressure_hydraulic` | c + d·load |
| `vibration_axial` | base + e·load^1.5, jitter grows with load |
| `vibration_radial` | base + f·load^1.2, jitter grows with load |
| `temperature_motor` | the physical state directly |
| `temperature_bearing` | the physical state directly |

All sensors return **0** in `OFF` (the gap-vs-zero decision is left to the publisher; here zeros make plots readable).

In [ ]:
@dataclass
class Machine:
    machine_id: str
    nominal_rpm: float = 3000.0
    ambient_c: float = 22.0

    # FSM state
    state: State = State.OFF
    state_elapsed_s: float = 0.0
    state_dwell_s: float = field(default_factory=lambda: pick_dwell(State.OFF))

    # Physical state
    load_actual: float = 0.0
    T_motor: float = 22.0
    T_bearing: float = 22.0

    # Time constants (seconds)
    tau_load_ramp: float = 5.0
    tau_load_steady: float = 30.0
    tau_T_motor: float = 180.0
    tau_T_bearing: float = 300.0

    # Sensor coefficients (tunable)
    k_current_a: float = 1.0
    k_current_b: float = 14.0
    k_power_factor: float = 0.42
    k_pressure_a: float = 80.0
    k_pressure_b: float = 70.0
    k_vib_axial_base: float = 0.10
    k_vib_axial_load: float = 0.25
    k_vib_radial_base: float = 0.15
    k_vib_radial_load: float = 0.35
    k_rpm_droop: float = 0.05

    def step(self, dt: float) -> None:
        self.state_elapsed_s += dt
        if self.state_elapsed_s >= self.state_dwell_s:
            self.state = pick_next_state(self.state)
            self.state_elapsed_s = 0.0
            self.state_dwell_s = pick_dwell(self.state)

        # Load: faster lag during RAMP_*; slower at steady state
        target = STATE_SPECS[self.state].target_load
        tau = self.tau_load_ramp if self.state in (
            State.RAMP_UP, State.RAMP_DOWN, State.STARTUP, State.SHUTDOWN
        ) else self.tau_load_steady
        alpha = 1.0 - math.exp(-dt / tau)
        self.load_actual += alpha * (target - self.load_actual)

        # Thermal: heating proportional to load², cooling toward ambient
        heat_input = 0.0 if self.state == State.OFF else 60.0 * (self.load_actual ** 2)
        T_target_motor = self.ambient_c + heat_input
        a_m = 1.0 - math.exp(-dt / self.tau_T_motor)
        self.T_motor += a_m * (T_target_motor - self.T_motor)

        T_target_bearing = self.ambient_c + 0.85 * (self.T_motor - self.ambient_c)
        a_b = 1.0 - math.exp(-dt / self.tau_T_bearing)
        self.T_bearing += a_b * (T_target_bearing - self.T_bearing)

    def sample(self) -> dict[str, float]:
        if self.state == State.OFF:
            return {k: 0.0 for k in (
                "temperature_motor", "temperature_bearing",
                "vibration_axial",   "vibration_radial",
                "current",           "spindle_rpm",
                "pressure_hydraulic","power",
            )}

        load = max(0.0, self.load_actual)
        jitter_axial  = np.random.normal(0, 0.02 + 0.05 * load)
        jitter_radial = np.random.normal(0, 0.03 + 0.07 * load)

        rpm      = self.nominal_rpm * (1.0 - self.k_rpm_droop * load) + np.random.normal(0, 8)
        current  = self.k_current_a + self.k_current_b * load + np.random.normal(0, 0.3)
        power    = self.k_power_factor * current * (1.0 + 0.1 * load) + np.random.normal(0, 0.2)
        pressure = self.k_pressure_a + self.k_pressure_b * load + np.random.normal(0, 1.0)
        vib_a    = self.k_vib_axial_base  + self.k_vib_axial_load  * load ** 1.5 + jitter_axial
        vib_r    = self.k_vib_radial_base + self.k_vib_radial_load * load ** 1.2 + jitter_radial

        return {
            "temperature_motor":   self.T_motor   + np.random.normal(0, 0.4),
            "temperature_bearing": self.T_bearing + np.random.normal(0, 0.3),
            "vibration_axial":     max(0.0, vib_a),
            "vibration_radial":    max(0.0, vib_r),
            "current":             max(0.0, current),
            "spindle_rpm":         max(0.0, rpm),
            "pressure_hydraulic":  max(0.0, pressure),
            "power":               max(0.0, power),
        }

## 4. Run a simulation

In [ ]:
def simulate(machine: Machine, duration_s: float, dt: float = 1.0) -> pd.DataFrame:
    """Run the machine forward and collect a tidy DataFrame (one row per dt)."""
    n_steps = int(duration_s / dt)
    rows: list[dict] = []
    for i in range(n_steps):
        machine.step(dt)
        sample = machine.sample()
        rows.append({
            "t_s":          i * dt,
            "state":        machine.state.value,
            "load":         machine.load_actual,
            "T_motor_phys": machine.T_motor,
            **sample,
        })
    return pd.DataFrame(rows)


random.seed(RNG_SEED); np.random.seed(RNG_SEED)
m = Machine(machine_id="M-001", state=State.OFF)
df = simulate(m, duration_s=2 * 3600, dt=1.0)
df.head()

## 5. Visual sanity checks

### 5.1 State + load timeline

State changes appear as colored bands. The `load` line should rise/fall smoothly during ramps and stay flat (with first-order relaxation) within a state.

In [ ]:
from matplotlib.patches import Patch

STATE_COLORS = {
    "OFF":              "#cccccc",
    "STARTUP":          "#ffd27f",
    "IDLE":             "#a8d8ea",
    "RAMP_UP":          "#ffb37f",
    "PRODUCTION_LIGHT": "#9ee09e",
    "PRODUCTION_HEAVY": "#3aa539",
    "RAMP_DOWN":        "#ff9999",
    "SHUTDOWN":         "#bbbbbb",
}

def shade_states(ax, df: pd.DataFrame) -> None:
    df = df.reset_index(drop=True)
    s = df["state"].to_numpy()
    starts = [0]
    for i in range(1, len(df)):
        if s[i] != s[i-1]:
            starts.append(i)
    starts.append(len(df))
    for a, b in zip(starts[:-1], starts[1:]):
        ax.axvspan(df["t_s"].iat[a], df["t_s"].iat[b-1],
                   color=STATE_COLORS.get(s[a], "#ffffff"), alpha=0.35)

fig, ax = plt.subplots(figsize=(14, 3.2))
shade_states(ax, df)
ax.plot(df["t_s"], df["load"], color="black", lw=1.2, label="load_actual")
ax.set_xlabel("t (s)"); ax.set_ylabel("load (0..1)")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Operating state (background) + load_actual")
ax.legend(handles=[Patch(facecolor=c, alpha=0.35, label=s) for s, c in STATE_COLORS.items()],
          ncol=4, loc="upper right", fontsize=8)
plt.show()

### 5.2 All 8 sensors

One subplot per sensor, sharing the x-axis. State bands repeated for context.

In [ ]:
SENSORS = [
    "temperature_motor", "temperature_bearing",
    "vibration_axial",   "vibration_radial",
    "current",           "spindle_rpm",
    "pressure_hydraulic","power",
]

fig, axes = plt.subplots(len(SENSORS), 1, figsize=(14, 1.6 * len(SENSORS)),
                         sharex=True)
for ax, sensor in zip(axes, SENSORS):
    shade_states(ax, df)
    ax.plot(df["t_s"], df[sensor], lw=0.8, color="navy")
    ax.set_ylabel(sensor, fontsize=8)
axes[-1].set_xlabel("t (s)")
fig.suptitle("8 sensor channels over time", y=0.995)
plt.tight_layout()
plt.show()

### 5.3 Pairwise consistency

These scatter plots verify the physical couplings. We exclude `OFF` samples to keep the plots readable.

In [ ]:
on = df[df["state"] != "OFF"]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
axes[0].scatter(on["current"], on["power"], s=4, alpha=0.4)
axes[0].set_xlabel("current (A)"); axes[0].set_ylabel("power (kW)")
axes[0].set_title("power ↔ current (≈ linear)")

axes[1].scatter(on["load"], on["spindle_rpm"], s=4, alpha=0.4)
axes[1].set_xlabel("load"); axes[1].set_ylabel("rpm")
axes[1].set_title("rpm droops with load")

axes[2].scatter(on["load"], on["vibration_radial"], s=4, alpha=0.4)
axes[2].set_xlabel("load"); axes[2].set_ylabel("vibration_radial (g)")
axes[2].set_title("vibration grows nonlinearly with load")

axes[3].scatter(on["T_motor_phys"], on["temperature_bearing"], s=4, alpha=0.4)
axes[3].set_xlabel("T_motor (°C)"); axes[3].set_ylabel("T_bearing (°C)")
axes[3].set_title("bearing slaved to motor (with lag)")

plt.tight_layout()
plt.show()

### 5.4 State-occupancy histogram

How much time the machine spent in each state. Useful to tune dwell distributions and transition weights.

In [ ]:
occ = (df["state"].value_counts(normalize=True) * 100).reindex(
    [s.value for s in State], fill_value=0.0
)
fig, ax = plt.subplots(figsize=(10, 3))
colors = [STATE_COLORS[s] for s in occ.index]
ax.bar(occ.index, occ.values, color=colors, edgecolor="black")
ax.set_ylabel("% of simulated time")
ax.set_title("State occupancy")
for i, v in enumerate(occ.values):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)
plt.xticks(rotation=20, ha="right")
plt.show()

## 6. Iteration playground

Tweak coefficients, re-run, re-plot. Re-run cell 4 to regenerate `df` after changing any constant in cell 3.

Things to play with:

- `STATE_SPECS[...].dwell_s` → how often regimes alternate.
- `Machine.tau_load_steady` → how stiff steady-state load is.
- `Machine.tau_T_motor` / `tau_T_bearing` → thermal inertia.
- `k_current_b`, `k_power_factor`, `k_vib_*` → sensor sensitivity.
- `Machine.k_rpm_droop` → how much spindle slows under load.

In [ ]:
# Example: faster thermal response, less stiff load — see more dynamics in 30 min
random.seed(RNG_SEED); np.random.seed(RNG_SEED)
m2 = Machine(machine_id="M-002", state=State.IDLE)
m2.tau_T_motor = 90.0
m2.tau_load_steady = 15.0
df2 = simulate(m2, duration_s=30 * 60, dt=1.0)

fig, ax = plt.subplots(figsize=(14, 3.2))
shade_states(ax, df2)
ax.plot(df2["t_s"], df2["load"], color="black", lw=1.2, label="load")
ax.plot(df2["t_s"], df2["T_motor_phys"] / 100, color="red",  lw=0.8, label="T_motor / 100")
ax.set_xlabel("t (s)"); ax.legend(loc="upper right")
ax.set_title("Fast-thermal variant (M-002, 30 min)")
plt.show()